---
title: "3.3.b Amenaza por erosión de orillas y dinámica fluvial"
short_title: "b. Amenaza"
subtitle: "Migración del cauce, erosión y sedimentación frente a Leticia, 1987–2024"
description: "Análisis de la dinámica del río Amazonas frente a Leticia con compuestos Landsat de aguas bajas y JRC Global Surface Water."
keywords: [erosión, sedimentación, dinámica fluvial, Landsat, MNDWI, Global Surface Water, Leticia]
---

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mromerot/leticia/blob/main/book/03_analisis/33_erosion/b_amenaza.ipynb)

In [ ]:
# Preparación del entorno. En Colab instala el paquete del proyecto desde GitHub (1–2 minutos);
# en una copia local usa el paquete ya instalado con `pip install -e .`.
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "leticia @ git+https://github.com/mromerot/leticia"], check=True)
from leticia.libro import preparar, intentar
preparar()

El Amazonas es un río de cauces múltiples que **erosiona su orilla cóncava y deposita sedimento en la convexa**.
Las islas se forman, migran y desaparecen en pocas décadas. Frente a Leticia, la isla de Santa Rosa surgió hace
unos 50 años por acumulación de sedimentos y el flujo se ha desplazado hacia el brazo peruano: según estudios de
la Universidad Nacional, la fracción del caudal que pasa por el lado colombiano bajó de ~30 % en 1993 a ~19,5 %
en 2025 [@rtvc2025sedimentacion]. Río abajo, en Puerto Nariño, la socavación destruyó el muelle en 2020
[@pigcct2022amazonas]. Este cuaderno mide esa dinámica con imágenes de satélite.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from leticia import aoi, espacial, exposicion, fuentes, viz
from leticia.config import parametros

pe = parametros()["erosion"]
urbana = aoi.area_urbana()
PIX_HA = (0.0003 * 111320) ** 2 * np.cos(np.radians(4.2)) / 1e4   # área de un píxel de 0,0003° en ha


def agua_anio(anio):
    ds = espacial.compuesto_landsat(anio, pe["meses_aguas_bajas"], ventana_anios=1 if anio < 2000 else 0)
    return (espacial.mndwi(ds) > pe["umbral_mndwi"]).rio.write_crs(4326), ds.attrs["escenas"]

## 1. El cauce en aguas bajas: 1987, 2001, 2013 y 2024

In [ ]:
anios = pe["anios_comparacion"]
mascaras, escenas = {}, {}
for a in anios:
    mascaras[a], escenas[a] = agua_anio(a)
fig, ax = plt.subplots(1, len(anios), figsize=(4.2 * len(anios), 4.4))
for a, anio in zip(ax, anios):
    viz.fondo(a, "bbox_rio", alfa=0.35)
    m = mascaras[anio]
    a.imshow(np.where(m.values, 1.0, np.nan), extent=(float(m.x.min()), float(m.x.max()), float(m.y.min()), float(m.y.max())),
             cmap="Blues", vmin=0, vmax=1.2, zorder=2, alpha=0.9)
    urbana.to_crs(4326).boundary.plot(ax=a, color="k", lw=0.7, zorder=3)
    viz.marco(a, "bbox_rio", f"{anio} ({escenas[anio]} escenas)")
fig.tight_layout()

In [ ]:
viz.pie(viz.nota_fuente("landsat") + " Agua: MNDWI > 0 (Xu, 2006) en la mediana de agosto–noviembre; mapa base Sentinel-2.")

## 2. Erosión y sedimentación entre 1987 y 2024

In [ ]:
a0, a1 = anios[0], anios[-1]
m0 = mascaras[a0].reindex_like(mascaras[a1], method="nearest")
m1 = mascaras[a1]
erosion = (~m0.astype(bool)) & m1.astype(bool)      # tierra que pasó a agua
deposito = m0.astype(bool) & (~m1.astype(bool))      # agua que pasó a tierra (islas, playas, bancos)
fig, ax = plt.subplots(figsize=(8, 6.4))
viz.fondo(ax, "bbox_rio", alfa=0.5)
ext = (float(m1.x.min()), float(m1.x.max()), float(m1.y.min()), float(m1.y.max()))
ax.imshow(np.where(m0.values & m1.values, 1.0, np.nan), extent=ext, cmap="Blues", vmin=0, vmax=1.5, zorder=2, alpha=0.6)
ax.imshow(np.where(erosion.values, 1.0, np.nan), extent=ext, cmap="Reds", vmin=0, vmax=1.1, zorder=3)
ax.imshow(np.where(deposito.values, 1.0, np.nan), extent=ext, cmap="YlOrBr", vmin=0, vmax=1.6, zorder=3)
urbana.to_crs(4326).boundary.plot(ax=ax, color="k", lw=0.8, zorder=4)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#2171b5", alpha=0.6, label="agua en ambos años"), Patch(color="#cb181d", label=f"tierra → agua (erosión)"),
                   Patch(color="#fec44f", label="agua → tierra (sedimentación)")], loc="lower left", fontsize=8, frameon=True)
viz.marco(ax, "bbox_rio", f"Cambios del cauce en aguas bajas, {a0}–{a1}")

In [ ]:
b = fuentes.bbox("bbox_rio")
sel = lambda da: da.sel(x=slice(b[0], b[2]), y=slice(b[3], b[1]))
res = pd.Series({"erosión (ha)": float(sel(erosion).sum()) * PIX_HA, "sedimentación (ha)": float(sel(deposito).sum()) * PIX_HA,
                 f"agua en aguas bajas {a0} (ha)": float(sel(m0).sum()) * PIX_HA, f"agua en aguas bajas {a1} (ha)": float(sel(m1).sum()) * PIX_HA})
display(res.round(0).to_frame("tramo frente a Leticia"))

:::{note} Nivel del río y cambio del cauce
Comparar dos años de aguas bajas mezcla dos efectos: el cambio de forma del cauce y la diferencia de nivel
(2024 fue la vaciante más baja del registro, así que aparecen más playas). Por eso el análisis se complementa con
las **transiciones** del JRC, que comparan la frecuencia del agua en 1984–1999 y 2000–2021 y no dependen de un
solo año.
:::

## 3. Transiciones del agua superficial, 1984–2021 (JRC)

In [ ]:
tr = fuentes.raster_estatico("jrc-gsw", "transitions")
CLASES = {1: ("permanente", "#0000ff"), 2: ("nueva permanente", "#22b14c"), 3: ("permanente perdida", "#d1102d"),
          4: ("estacional", "#99d9ea"), 5: ("nueva estacional", "#b5e61d"), 6: ("estacional perdida", "#e6a1aa"),
          7: ("estacional → permanente", "#ff7f27"), 8: ("permanente → estacional", "#ffc90e"),
          9: ("permanente efímera", "#7f7f7f"), 10: ("estacional efímera", "#c3c3c3")}
from matplotlib.colors import ListedColormap, BoundaryNorm
cods = list(CLASES)
cmap = ListedColormap([CLASES[k][1] for k in cods])
norm = BoundaryNorm(np.arange(0.5, 11.5), cmap.N)
t = sel(tr)
fig, ax = plt.subplots(figsize=(8, 6.4))
viz.fondo(ax, "bbox_rio", alfa=0.4)
ax.imshow(np.where(t.values > 0, t.values, np.nan), extent=(float(t.x.min()), float(t.x.max()), float(t.y.min()), float(t.y.max())),
          cmap=cmap, norm=norm, interpolation="nearest", zorder=2)
urbana.to_crs(4326).boundary.plot(ax=ax, color="k", lw=0.8, zorder=3)
ax.legend(handles=[Patch(color=CLASES[k][1], label=CLASES[k][0]) for k in cods if (t.values == k).any()], loc="lower left", fontsize=7, frameon=True)
viz.marco(ax, "bbox_rio", "Transiciones del agua superficial (JRC GSW)")
px_tr = (float(abs(t.x[1] - t.x[0])) * 111320) ** 2 * np.cos(np.radians(4.2)) / 1e4
conteo = pd.Series(t.values.ravel()).value_counts()
display((conteo.rename(index={k: v[0] for k, v in CLASES.items()}).drop(0, errors="ignore") * px_tr).round(0).to_frame("ha").head(10))

In [ ]:
viz.pie(viz.nota_fuente("agua_superficial_jrc") + " Pekel et al. (2016).")

**Lectura.** Las clases **«permanente perdida»** (rojo) y **«nueva permanente»** (verde) marcan dónde el cauce
migró: el río abandonó unas zonas y ocupó otras. Las **«nuevas estacionales»** suelen ser playas y bancos de
arena que emergen en aguas bajas. Frente a Leticia, la acumulación de sedimento en el brazo colombiano y la
erosión en la margen opuesta son consistentes con el desplazamiento del flujo hacia el brazo peruano
[@rtvc2025sedimentacion].